In [2]:
import pandas as pd
import geopandas as gpd


In [3]:
# import tokyo_slim_pop.geojson and vacancy.geojson
tokyo = gpd.read_file('tokyo_slim_pop.geojson')
vacancy = gpd.read_file('vacancy.geojson')

In [ ]:
tokyo

In [5]:
# identify columns in tokyo slim pop data that start with a 4 digit year
year_columns = [col for col in tokyo.columns if col[:4].isdigit()]

print(year_columns)

# identfiy columns in tokyo slim pop data that do not start with year to keep as is
non_year_columns = [col for col in tokyo.columns if col not in year_columns]
print(non_year_columns)


['2023_building_count', '2023_office_count', '2023_office_total_use_area', '1996_building_count', '2001_building_count', '2006_building_count', '2011_building_count', '2016_building_count', '2023_old__building_count', '1996_office_count', '2001_office_count', '2006_office_count', '2011_office_count', '2016_office_count', '2023_old__office_count', '1996_office_total_use_area', '2001_office_total_use_area', '2006_office_total_use_area', '2011_office_total_use_area', '2016_office_total_use_area', '2023_old__office_total_use_area', '1996_housing_total_use_area', '2001_housing_total_use_area', '2006_housing_total_use_area', '2011_housing_total_use_area', '2016_housing_total_use_area', '2023_housing_total_use_area', '1996_other_total_use_area', '2001_other_total_use_area', '2006_other_total_use_area', '2011_other_total_use_area', '2016_other_total_use_area', '2023_other_total_use_area']
['KEY_CODE', 'diff_building_count', 'diff_office_count', 'diff_office_total_use_area', 'population', 'pop_

In [12]:
# pivot tokyo slim pop data by keeping non year columns, creating new rows for distinct KEY_CODE and YEAR, along with all non year columns. 
# then, create new columns of all the year columns, but take away the year prefix and underscore from the column name
# finally, add the values from the year columns into the new VALUE column
tokyo_melted = tokyo.melt(id_vars=non_year_columns, value_vars=year_columns, var_name='YEAR', value_name='VALUE')
tokyo_melted['YEAR'] = tokyo_melted['YEAR'].str[:4]
tokyo_melted.head()

# add the year columns (eg 2023_building_count, etc) without the prefix and underscore as new columns (eg building_count, etc), 
# then add the appropriate values, ie, for 2023_building_count, add the values to building_count column where YEAR is 2023
for col in year_columns:
    new_col = col[5:]  # remove the year prefix and underscore
    tokyo_melted[new_col] = tokyo_melted.apply(lambda row: row['VALUE'] if row['YEAR'] == col[:4] else None, axis=1)

tokyo_melted.head()

# ensure YEAR is a string for downstream joins
tokyo_melted['YEAR'] = tokyo_melted['YEAR'].astype(str)


,KEY_CODE,diff_building_count,diff_office_count,diff_office_total_use_area,population,pop_male,pop_female,pop_0_14,pop_15_64,pop_65_plus,...,YEAR,VALUE,building_count,office_count,office_total_use_area,old__building_count,old__office_count,old__office_total_use_area,housing_total_use_area,other_total_use_area
0,533925552,NaN,NaN,NaN,3301.0,1716.0,1585.0,670.0,2205.0,406.0,...,2023,2.0,NaN,NaN,NaN,2.0,2.0,2.0,2.0,2.0
1,533925554,NaN,NaN,NaN,3665.0,1882.0,1783.0,500.0,2190.0,949.0,...,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,533925561,135.0,6.0,9587.10453,4211.0,2061.0,2150.0,575.0,2456.0,1098.0,...,2023,664.0,NaN,NaN,NaN,664.0,664.0,664.0,664.0,664.0
3,533925562,104.0,2.0,-18352.24181,6385.0,3334.0,3051.0,627.0,3945.0,1596.0,...,2023,1071.0,NaN,NaN,NaN,1071.0,1071.0,1071.0,1071.0,1071.0
4,533925563,72.0,0.0,-571.34694,1731.0,890.0,841.0,246.0,1066.0,364.0,...,2023,557.0,NaN,NaN,NaN,557.0,557.0,557.0,557.0,557.0


In [13]:
# rename vacancy Code to KEY_CODE and align YEAR dtype for merge
vacancy = vacancy.rename(columns={'Code': 'KEY_CODE'})
vacancy['YEAR'] = vacancy['YEAR'].astype(str)

# keep only data columns needed for the join (drop geometry)
vacancy_fields = ['KEY_CODE', 'YEAR', '空室率', '空室率分散', 'RENTABLE_AREA_NET_SUM', 'KEIYAKUMENSEKITUBO_SUM', 'FLG']
vacancy_for_join = vacancy[vacancy_fields]
vacancy_for_join.head()


,KEY_CODE,YEAR,空室率,空室率分散,RENTABLE_AREA_NET_SUM,KEIYAKUMENSEKITUBO_SUM,FLG
0,533925454,2012,0.067770,0.102441,NaN,NaN,0.0
1,533925461,2012,0.117405,0.153311,NaN,NaN,0.0
2,533925462,2012,0.099054,0.115535,NaN,NaN,0.0
3,533925463,2012,0.055977,0.092511,NaN,NaN,0.0
4,533925464,2012,0.081717,0.092739,NaN,NaN,0.0


In [14]:
# join vacancy data onto the melted tokyo dataset on KEY_CODE and YEAR
tokyo_with_vacancy = tokyo_melted.merge(vacancy_for_join, how='left', on=['KEY_CODE', 'YEAR'])
tokyo_with_vacancy.head()


,KEY_CODE,diff_building_count,diff_office_count,diff_office_total_use_area,population,pop_male,pop_female,pop_0_14,pop_15_64,pop_65_plus,...,old__building_count,old__office_count,old__office_total_use_area,housing_total_use_area,other_total_use_area,空室率,空室率分散,RENTABLE_AREA_NET_SUM,KEIYAKUMENSEKITUBO_SUM,FLG
0,533925552,NaN,NaN,NaN,3301.0,1716.0,1585.0,670.0,2205.0,406.0,...,2.0,2.0,2.0,2.0,2.0,0.084680,0.067670,NaN,NaN,0.0
1,533925554,NaN,NaN,NaN,3665.0,1882.0,1783.0,500.0,2190.0,949.0,...,NaN,NaN,NaN,NaN,NaN,0.066369,0.067564,NaN,NaN,0.0
2,533925561,135.0,6.0,9587.10453,4211.0,2061.0,2150.0,575.0,2456.0,1098.0,...,664.0,664.0,664.0,664.0,664.0,0.054879,0.067623,NaN,NaN,0.0
3,533925562,104.0,2.0,-18352.24181,6385.0,3334.0,3051.0,627.0,3945.0,1596.0,...,1071.0,1071.0,1071.0,1071.0,1071.0,0.054742,0.067662,NaN,NaN,0.0
4,533925563,72.0,0.0,-571.34694,1731.0,890.0,841.0,246.0,1066.0,364.0,...,557.0,557.0,557.0,557.0,557.0,0.038363,0.067504,NaN,NaN,0.0


In [16]:
# export the merged dataset as GeoJSON
from pathlib import Path
import json

# rebuild as GeoDataFrame to keep geometry/CRS, then convert to WGS84
geo_tokyo_with_vacancy = gpd.GeoDataFrame(tokyo_with_vacancy, geometry='geometry', crs=tokyo.crs)
geo_tokyo_with_vacancy = geo_tokyo_with_vacancy.to_crs(epsg=4326)

# write strict GeoJSON (drop CRS member for RFC7946 validators)
geojson_obj = json.loads(geo_tokyo_with_vacancy.to_json())
geojson_obj.pop('crs', None)
Path('tokyo_with_vacancy.geojson').write_text(json.dumps(geojson_obj, ensure_ascii=False))


75498544